# M.I.N.E.R.V.A.

Tagalog educational misinformation card pipeline. See [README.md](../README.md), [docs/CODEBOOK.md](../docs/CODEBOOK.md), and [docs/architecture.md](../docs/architecture.md) for full documentation.

**Runtime:** Colab A100 + High-RAM (~30 min). Set in Runtime → Change runtime type.

## 1. Configuration

In [5]:
# ============================================================
# v2.9.0 pipeline configuration
# (v2.9.0 = v2.8.7 +
#         + scripts/35 NEW: place-name pseudonymization. Audit found 4 PH cities
#           leaking through the existing person-only pseudonymizer.
#         + templates/places_blocklist.txt: 17 regions, 80+ provinces,
#           80+ cities, common landmarks. Deterministic City W/X/Y/Z mapping.
#         + templates/response_bank_v2.json: 225 hand-shaped phrases
#           (72%% diversity) covering all 12 indicators × 3 tiers × 2 verdicts.
#           Replaces v2.8.7's stub explanation block. Faithfulness audit went
#           from 87.18%% (v2.8.7) → projected 98%%+.
#         + scripts/29 update: indicator-coverage filter pre-merge. GPT-2 cards
#           are dropped if their fired indicators don't have a response-bank
#           entry, eliminating the failure mode that caused 231 mismatches.
#         + scripts/37 NEW: held-out detector evaluation on hand-labeled
#           GPT-2 cards. Replaces the tautological 'detector accuracy on the
#           pool' metric with a real generalization claim.
#         + scripts/10b update: report stamps v2.9.0 + adds an audit assertion
#           that loudly warns if percentile binning didn't activate (the
#           failure mode that left v2.8.7's GPT-2 conditioning unlearnable).
# (v2.8.7 = v2.8.6 +
#         + scripts/29 NEW: merges GPT-2 cards into the template stream so they
#           reach the pool (previous pipeline had no merge step — GPT-2 outputs
#           were stranded in jsonl files no script ever read).
#         + Persistent-generation loop: each label retries with new seeds until
#           GPT2_MIN_PROMOTED_PER_LABEL cards survive merge, capped at
#           GPT2_MAX_ATTEMPTS to prevent infinite loops.
#         + Sentence-recovery: trim mid-word truncations back to last '.', '!',
#           '?' so usable Tagalog isn't thrown away.
#         + Code remap: GPT-2 outputs Excel-style codes from JCBlaise pseudo
#           ('Candidate AA', 'Candidate DKR'); script 29 remaps them to A/B/C
#           in order-of-first-appearance so cards conform to the game's allowlist.
#         + max_new_tokens raised 120 -> 200 to reduce truncation upstream.)
# (v2.8.6 = v2.8.5 + percentile-based control-token binning in 10b
#           (so GPT-2 can actually learn graph/qlat/ensem/tier conditioning;
#           previous fixed 0.6/0.8 thresholds put 96% of training in 'high')
#         + GPT2_EPOCHS bumped 3 -> 8 (3-epoch run only converged loss 3.49)
#         + differentiated seeds for fake/real GPT-2 generation
#         + save cell now bundles models/qlattice_equation.txt + best_qlattice.json
#         + cumulative: includes v2.8.4 feyn pin + script 08 fallback)
# ============================================================

# Repo
GITHUB_USER   = "robertgeraldsenasin"
GITHUB_REPO   = "MINERVA"
GITHUB_BRANCH = "main"

# ----------------------------------------------------------------
# PIPELINE MODE — pick ONE of:
#
#   "templates_only"  : skip detector training AND GPT-2 generation
#                       (~3 minutes, no GPU needed). Pool comes from
#                       deterministic templates only. Auto-disables
#                       USE_GPT2.
#
#   "full_cached"     : skip detector training IF the artifacts are
#                       cached locally or in Drive; otherwise train.
#                       GPT-2 still runs to produce neuro-symbolic
#                       generations.
#
#   "full_retrain"    : ALWAYS retrain detectors AND run GPT-2
#                       neuro-symbolic generation (~50-65 min on T4).
#                       v2.8 DEFAULT — blank-slate testing realizes
#                       the FULL architecture per BATB SO 1, 2, 4
#                       every time.
# ----------------------------------------------------------------
PIPELINE_MODE = "full_retrain"

# Drive (set False to skip Drive entirely; pipeline still works,
# falls back to a downloadable zip)
USE_DRIVE        = True
DRIVE_OUTPUT_DIR = "MINERVA_outputs"

# Detector training (Mode B — used in full_* modes)
TRAIN_RUN_ID     = "v28_panel"        # tag for this training run
TRAIN_SEEDS      = "13,29,47,89,127"  # v2.9.3: 5 prime seeds (Liu 2019 RoBERTa protocol)
TRAIN_EPOCHS     = 3                  # default per scripts/04, 05
TRAIN_MAX_LEN    = 256

# ----------------------------------------------------------------
# GPT-2 NEURO-SYMBOLIC GENERATION (BATB Specific Objective 2)
#
# When True, runs scripts 10b/11b/12b — neuro-symbolic conditioning
# that makes GPT-2 generation reflect ALL upstream signals (label +
# DE-GNN graph confidence + QLattice symbolic-equation score +
# detector ensemble + teaching tier) via 18 control tokens.
#
# This realizes the paper's "rule-constrained content generation"
# claim. Auto-disabled when PIPELINE_MODE = "templates_only".
# Requires GPU. Adds ~15-20 minutes.
# ----------------------------------------------------------------
USE_GPT2         = True
GPT2_BASE_MODEL  = "jcblaise/gpt2-tagalog"
GPT2_EPOCHS      = 8        # v2.9.0: was 3 (insufficient — 80s training, eval loss only reached 3.35)

# v2.9.0: with percentile binning verified active and response-bank coverage
# filter, expected GPT-2 promotion rate jumps from ~10%% (v2.8.7) to ~30-40%%.
# Adjust GPT2_MIN_PROMOTED_PER_LABEL up if you want a denser pool.

# v2.9.0: persistent-generation knobs. The GPT-2 path generates batches
# of cards, runs each through scoring + script 29's filters (sentence
# recovery, candidate code remap, candidate-allowlist), and retries with
# new seeds until GPT2_MIN_PROMOTED_PER_LABEL cards survive — capped at
# GPT2_MAX_ATTEMPTS so it can't loop forever.
GPT2_MIN_PROMOTED_PER_LABEL = 100   # target survivors per label
GPT2_MAX_ATTEMPTS           = 4     # max generation batches per label
GPT2_GEN_POOL_SIZE          = 500   # v2.9.3 renamed from GPT2_BATCH_SIZE
GPT2_BATCH_SIZE             = GPT2_GEN_POOL_SIZE  # back-compat alias
GPT2_MAX_NEW_TOKENS         = 200   # was 120 — fewer mid-word cutoffs
GPT2_LR          = 5e-5

# Template-mode pipeline knobs (used in ALL modes)
N_PER_TEMPLATE   = 18
TARGET_POOL      = 700
DAYS_IN_DECK     = 7
CARDS_PER_DAY    = 8
MIN_CREDIBLE_PER_DAY = 3
RNG_SEED         = 1729

# Pre-pilot pack
PILOT_N          = 30
PILOT_SEED       = 1729

# v2.8 build identifier
MINERVA_VERSION  = "v2.9.9"

print(f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f" M.I.N.E.R.V.A. {MINERVA_VERSION} — Configuration")
print(f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"  PIPELINE_MODE = {PIPELINE_MODE!r}")
print(f"  USE_GPT2      = {USE_GPT2}")
print(f"  USE_DRIVE     = {USE_DRIVE}")
print(f"  TRAIN_SEEDS   = {TRAIN_SEEDS}")
print(f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 M.I.N.E.R.V.A. v2.9.9 — Configuration
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  PIPELINE_MODE = 'full_retrain'
  USE_GPT2      = True
  USE_DRIVE     = True
  TRAIN_SEEDS   = 13,29,47,89,127
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


## 2. Mount Google Drive (gracefully — skips if unavailable)

In [6]:
DRIVE_MOUNT_OK = False
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_MOUNT_OK = True
        print("✓ Drive mounted at /content/drive")
    except (ImportError, RuntimeError, ValueError) as e:
        print(f"⚠ Drive not available ({type(e).__name__}); continuing without it")
        print("  (this is normal outside Colab or in incognito)")
else:
    print("⚠ USE_DRIVE=False — skipping Drive mount")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Drive mounted at /content/drive


## 3. Clone the repo

In [7]:
import os, shutil, subprocess

REPO_PATH = "/content/MINERVA"

# CRITICAL: chdir to a known-existing parent BEFORE any rmtree.
# Colab keeps the kernel's cwd between runs. If a previous cell did
# os.chdir(REPO_PATH) and we then rmtree(REPO_PATH), the kernel ends
# up sitting in a deleted directory and `git clone` fails with
# "Unable to read current working directory".
os.chdir("/content")

if os.path.exists(REPO_PATH):
    print(f"⚠ {REPO_PATH} already exists — removing and re-cloning")
    shutil.rmtree(REPO_PATH)

clone_url = f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
print(f"Cloning {clone_url} (branch: {GITHUB_BRANCH})...")
result = subprocess.run(
    ["git", "clone", "-b", GITHUB_BRANCH, "--depth", "1", clone_url, REPO_PATH],
    capture_output=True, text=True
)
if result.returncode != 0:
    print(f"✗ Clone failed:\n{result.stderr}")
    raise RuntimeError("git clone failed")

os.chdir(REPO_PATH)
print(f"✓ Repo at {REPO_PATH}")
print(f"  Branch: {subprocess.check_output(['git','branch','--show-current'],text=True).strip()}")
print(f"  HEAD  : {subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip()}")


Cloning https://github.com/robertgeraldsenasin/MINERVA.git (branch: main)...
✓ Repo at /content/MINERVA
  Branch: main
  HEAD  : 27596d4


## 4. Verify required files arrived

In [8]:
from pathlib import Path

required = [
    "scripts/candidate_config.py",            # ⭐ EDITABLE
    "scripts/30_template_scenario_generator.py",
    "scripts/31_universal_pseudonymize.py",
    "scripts/32_validate_detectors_on_templates.py",
    "scripts/40_export_pilot_pack.py",
    "scripts/26_faithfulness_audit.py",
    "scripts/21_balance_unity_cards.py",
    "scripts/23_enforce_election_theme.py",
    "scripts/24_curate_teaching_cards.py",
    "scripts/28_draw_user_deck.py",
    "scripts/minerva_candidates.py",
    "scripts/minerva_filters.py",
    "scripts/minerva_schemas.py",
    "scripts/minerva_indicators.py",
    "scripts/minerva_response_bank.py",
    "tests/test_filters.py",
    "tests/test_pilot_pack.py",
    "requirements.txt",
]

print("Required files in repo:")
all_ok = True
for f in required:
    ok = Path(f).exists()
    flag = "✓" if ok else "✗ MISSING"
    print(f"  {flag}  {f}")
    if not ok:
        all_ok = False

if not all_ok:
    raise RuntimeError("Missing files — patch may not have been pushed yet. Check the branch.")

# Show current candidate config
import sys
sys.path.insert(0, "scripts")
import candidate_config
print()
print("Candidate config (edit scripts/candidate_config.py to change):")
for c in candidate_config.CANDIDATES_CONFIG:
    print(f"  {c['code']}: {candidate_config.full_name(c)} ({c['archetype']}, {c['region']})")

print()
print("✓ All v2.6-final-consolidated files present.")


Required files in repo:
  ✓  scripts/candidate_config.py
  ✓  scripts/30_template_scenario_generator.py
  ✓  scripts/31_universal_pseudonymize.py
  ✓  scripts/32_validate_detectors_on_templates.py
  ✓  scripts/40_export_pilot_pack.py
  ✓  scripts/26_faithfulness_audit.py
  ✓  scripts/21_balance_unity_cards.py
  ✓  scripts/23_enforce_election_theme.py
  ✓  scripts/24_curate_teaching_cards.py
  ✓  scripts/28_draw_user_deck.py
  ✓  scripts/minerva_candidates.py
  ✓  scripts/minerva_filters.py
  ✓  scripts/minerva_schemas.py
  ✓  scripts/minerva_indicators.py
  ✓  scripts/minerva_response_bank.py
  ✓  tests/test_filters.py
  ✓  tests/test_pilot_pack.py
  ✓  requirements.txt
  ✗ MISSING  CLAUDE.md


RuntimeError: Missing files — patch may not have been pushed yet. Check the branch.

## 5. Install dependencies (~30s)

In [ ]:
# v2.9.4 install — uses requirements.txt with Colab-compatible pins.
#
# v2.9.4 fixed: requirements.txt's numpy pin relaxed from <2.0 to <3.0 so pip
# doesn't fight Colab's preinstalled numpy 2.x. No kernel restart needed.
#
# feyn has no version pin in requirements.txt (per user request).
# pip resolves to the latest published version (currently 3.5.0).
#
# CRITICAL: do NOT subsequently install fsspec or datasets manually —
# Colab's defaults will overwrite the pinned versions and re-introduce
# the LocalFileSystem NotImplementedError that broke v2.7 runs.

import subprocess, sys

print("Installing v2.9.4 dependencies (this takes ~1-2 min)...")
result = subprocess.run(
    ["pip", "install", "-q", "--upgrade", "pip"],
    capture_output=False, check=False
)
result = subprocess.run(
    ["pip", "install", "-q", "-r", "requirements.txt"],
    capture_output=False, check=False
)

# Verify the critical packages have correct versions
print("\nVerifying versions:")
import importlib

def check(pkg, attr="__version__"):
    try:
        m = importlib.import_module(pkg)
        v = getattr(m, attr, "?")
        print(f"  {pkg}: {v}")
        return v
    except Exception as e:
        print(f"  {pkg}: NOT INSTALLED ({e})")
        return None

fsspec_v = check("fsspec")
datasets_v = check("datasets")
transformers_v = check("transformers")
huggingface_hub_v = check("huggingface_hub")
torch_v = check("torch")
sklearn_v = check("sklearn")
feyn_v = check("feyn")

# Critical compatibility check: fsspec must be <= 2024.6.1
if fsspec_v:
    parts = [int(x) for x in fsspec_v.replace("rc", ".").split(".")[:3] if x.isdigit()]
    year, month = parts[0], parts[1] if len(parts) > 1 else 0
    if (year, month) > (2024, 6):
        print(f"\n⚠ WARNING: fsspec {fsspec_v} may be incompatible with datasets.")
        print(f"  Run this to force-fix:")
        print(f"  !pip install --force-reinstall 'fsspec>=2023.10.0,<=2024.6.1' 'datasets>=2.14,<3.0'")

print("\n✓ Install verification complete.")


## 6. Working folders

In [ ]:
for d in ["generated", "reports", "generated/decks", "reports/pilot_pack"]:
    Path(d).mkdir(parents=True, exist_ok=True)
print("✓ Folders ready: generated/, reports/, generated/decks/, reports/pilot_pack/")


## 6b. Environment capture (reproducibility)

In [ ]:
# v2.9.3: capture run environment for reproducibility report
import json, sys, os, platform
from pathlib import Path

env_info = {
    "timestamp": __import__('datetime').datetime.utcnow().isoformat() + 'Z',
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "minerva_version": MINERVA_VERSION,
    "pipeline_mode": PIPELINE_MODE,
}

# GPU detection
GPU_NAME = ""
try:
    import torch
    env_info["torch"] = torch.__version__
    if torch.cuda.is_available():
        GPU_NAME = torch.cuda.get_device_name(0)
        env_info["gpu"] = GPU_NAME
        env_info["cuda"] = torch.version.cuda
        try:
            props = torch.cuda.get_device_properties(0)
            env_info["gpu_total_memory_gb"] = round(props.total_memory / 1e9, 1)
        except Exception: pass
    else:
        env_info["gpu"] = "none — CPU only"
except ImportError:
    env_info["torch"] = "not installed"

# v2.9.3: derive GPU-aware batch sizes
_g = GPU_NAME.upper()
if "A100" in _g and "80" in _g:
    GPT2_TRAIN_BATCH = 16
    DETECTOR_BATCH, GRAD_ACCUM = 32, 1
elif "A100" in _g:
    GPT2_TRAIN_BATCH = 8
    DETECTOR_BATCH, GRAD_ACCUM = 32, 1
elif "V100" in _g:
    GPT2_TRAIN_BATCH = 4
    DETECTOR_BATCH, GRAD_ACCUM = 16, 2
else:  # T4 or unknown
    GPT2_TRAIN_BATCH = 2
    DETECTOR_BATCH, GRAD_ACCUM = 8, 4
env_info["derived_batch"] = {
    "gpt2_train_batch": GPT2_TRAIN_BATCH,
    "detector_batch": DETECTOR_BATCH,
    "grad_accum": GRAD_ACCUM,
    "detector_effective_batch": DETECTOR_BATCH * GRAD_ACCUM,
}

# Persist (only if /content/MINERVA exists; runs after clone)
_rd = Path("reports")
if _rd.exists():
    _rd.mkdir(exist_ok=True)
    with open(_rd / "_environment.json", "w") as f:
        json.dump(env_info, f, indent=2)
    print(f"✓ Wrote reports/_environment.json")

for k, v in env_info.items():
    if isinstance(v, dict):
        print(f"  {k}:")
        for sk, sv in v.items(): print(f"    {sk}: {sv}")
    else:
        print(f"  {k}: {v}")


## 7. Run unit tests (smoke check)

Expected: (40 prior + 19 strict_allowlist + 25 neurosymbolic_corpus + 27 qlattice_evaluator + 14 degnn_graph).

In [ ]:
import subprocess
result = subprocess.run(
    ["python", "-m", "pytest", "tests/", "-q"],
    capture_output=True, text=True
)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
print(result.stderr[-500:] if result.stderr else '')

# Expected: 259 passed (40 prior + 19 strict_allowlist + 25 neurosymbolic_corpus
# + 27 qlattice_evaluator + 14 degnn_graph)
if result.returncode != 0:
    raise SystemExit(f'Tests failed: {result.returncode}')
print('✓ All tests pass — proceeding to pipeline')


## 7a. Pre-flight: verify dataset download

In [ ]:
# Pre-flight: run script 01 and validate its output.
import subprocess, sys
from pathlib import Path
import pandas as pd

if PIPELINE_MODE == "templates_only":
    print("⏭️  Skipping pre-flight (templates_only mode — no dataset needed).")
else:
    print("Running scripts/01_download_dataset.py ...")
    res = subprocess.run([sys.executable, "scripts/01_download_dataset.py"],
                         capture_output=False)
    if res.returncode != 0:
        raise SystemExit(
            f"✗ Script 01 failed with exit code {res.returncode}. "
            "Read the FATAL block above for diagnosis. Do NOT proceed to 7b."
        )

    csv_path = Path("data/raw/jcblaise.csv")
    assert csv_path.exists(), f"✗ Expected output {csv_path} was not created"

    df = pd.read_csv(csv_path)
    assert "text" in df.columns or "article" in df.columns, \
        f"✗ Missing text column. Got: {list(df.columns)}"
    assert "label" in df.columns, \
        f"✗ Missing label column. Got: {list(df.columns)}"

    n_rows = len(df)
    label_counts = dict(df["label"].value_counts().sort_index())
    print(f"\n✓ Pre-flight PASSED:")
    print(f"  rows         : {n_rows}")
    print(f"  columns      : {list(df.columns)}")
    print(f"  label dist.  : {label_counts}")
    print(f"  expected ~3,206 rows, roughly balanced 0/1")
    if not (3000 <= n_rows <= 3500):
        print(f"  ⚠  Row count {n_rows} is outside expected range "
              f"— inspect the CSV before continuing.")


## 7b. Detector training pipeline (scripts 01-17)

This block runs the upstream stages that the rest of the pipeline depends on:

In [ ]:
# Detector training — gated by PIPELINE_MODE
import os, shutil, subprocess, sys
from pathlib import Path

REQUIRED_ARTIFACTS = [
    "data/processed/train.csv",
    "data/processed/val.csv",
    "data/processed/test.csv",
    "data/features/train_tabular.csv",
    "data/features/val_tabular.csv",
    "data/features/test_tabular.csv",
    "data/features/degnn_preds.csv",
    "models/qlattice_equation.txt",
    "models/roberta_finetuned/config.json",
    "models/distilbert_multilingual_finetuned/config.json",
]

# Clear stale env from any previous run
os.environ.pop("_NEED_TRAIN", None)

def _run_step(cmd, expected_outputs):
    """Run a pipeline script and verify it produced expected outputs.

    v2.9.0: Streams output line-by-line and NEVER raises SystemExit.
    SystemExit triggers a Colab/IPython 3.11 bug that corrupts traceback
    rendering with: "AttributeError: 'tuple' object has no attribute 'f_lineno'"
    and hides the actual error from the script. We use Popen + line streaming
    + RuntimeError instead, so the real failure is always visible.

    cmd: list of strings, e.g. ["python", "scripts/01_download_dataset.py"]
    expected_outputs: list of file paths that must exist after the script runs
    """
    print(f"  ▶ Running: {' '.join(cmd)}", flush=True)

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,   # merge so order is preserved
        text=True,
        bufsize=1,                  # line-buffered
    )
    output_lines = []
    try:
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="", flush=True)
            output_lines.append(line)
        proc.wait()
    except KeyboardInterrupt:
        proc.kill()
        proc.wait()
        raise

    rc = proc.returncode
    if rc != 0:
        print(f"\n{'='*60}")
        print(f"✗ STEP FAILED with exit code {rc}: {' '.join(cmd)}")
        print(f"{'='*60}")
        if output_lines:
            print("Last 40 lines of output (look here for the actual error):")
            print("-" * 60)
            for line in output_lines[-40:]:
                print(line, end="")
            print("-" * 60)
        else:
            print("(no output captured — script crashed before writing anything)")
        # RuntimeError, NOT SystemExit. SystemExit triggers the Colab/IPython
        # 3.11 traceback-rendering bug that hides the real error.
        raise RuntimeError(
            f"Pipeline aborted at: {cmd[1] if len(cmd) > 1 else cmd[0]} "
            f"(exit {rc}). See output above for diagnosis."
        )

    missing = [p for p in expected_outputs if not Path(p).exists()]
    if missing:
        print(f"\n✗ STEP COMPLETED BUT EXPECTED OUTPUTS MISSING:")
        for m in missing:
            print(f"    - {m}")
        raise RuntimeError(f"Output verification failed for: {' '.join(cmd)}")

    if expected_outputs:
        print(f"  ✓ Verified {len(expected_outputs)} output(s)")

def _try_restore_from_drive():
    """If Drive is mounted and a previous run cached artifacts, restore them."""
    if not DRIVE_MOUNT_OK:
        return False
    cache_dir = Path("/content/drive/MyDrive") / DRIVE_OUTPUT_DIR / "detector_artifacts"
    if not cache_dir.exists():
        return False
    print(f"  Restoring detector artifacts from Drive: {cache_dir}")
    for sub in ("data", "models", "logs"):
        src = cache_dir / sub
        if src.exists():
            dst = Path(sub)
            dst.mkdir(parents=True, exist_ok=True)
            for item in src.rglob("*"):
                if item.is_file():
                    rel = item.relative_to(src)
                    target = dst / rel
                    target.parent.mkdir(parents=True, exist_ok=True)
                    if not target.exists():
                        shutil.copy2(item, target)
    return True

def _save_to_drive():
    """Cache artifacts to Drive for the next session."""
    if not DRIVE_MOUNT_OK:
        return
    cache_dir = Path("/content/drive/MyDrive") / DRIVE_OUTPUT_DIR / "detector_artifacts"
    cache_dir.mkdir(parents=True, exist_ok=True)
    for sub in ("data", "models", "logs"):
        if Path(sub).exists():
            shutil.copytree(sub, cache_dir / sub, dirs_exist_ok=True)
    print(f"  ✓ Cached artifacts to Drive: {cache_dir}")

def _artifacts_present():
    return all(Path(p).exists() for p in REQUIRED_ARTIFACTS)

if PIPELINE_MODE == "templates_only":
    print("PIPELINE_MODE = templates_only — skipping detector training.")
    print("  Mode A (templates) does not require detector artifacts.")
    print("  GPT-2 path will be auto-disabled.")
    USE_GPT2 = False
elif PIPELINE_MODE == "full_cached":
    print("PIPELINE_MODE = full_cached — checking for cached artifacts...")
    if _artifacts_present():
        print("  ✓ All required artifacts present locally; skipping training.")
    else:
        if _try_restore_from_drive() and _artifacts_present():
            print("  ✓ Restored artifacts from Drive; skipping training.")
        else:
            print("  ⏵ No cache — running full detector training pipeline below.")
            os.environ["_NEED_TRAIN"] = "1"
elif PIPELINE_MODE == "full_retrain":
    print("PIPELINE_MODE = full_retrain — forcing retrain.")
    os.environ["_NEED_TRAIN"] = "1"
else:
    raise ValueError(f"Unknown PIPELINE_MODE: {PIPELINE_MODE!r}")


### 7b.1 Data preparation (scripts 01 / 02 / 03)

Downloads JCBlaise, normalizes text, splits 70/15/15.

In [ ]:
if os.environ.get("_NEED_TRAIN") == "1":
    _run_step(["python", "scripts/01_download_dataset.py"],
              expected_outputs=["data/raw/jcblaise.csv"])
    # Script 02 writes data/processed/corpus.csv (NOT jcblaise_clean.csv —
    # that was a v2.8 path-mismatch bug fixed in v2.9.0).
    _run_step(["python", "scripts/02_prepare_dataset.py"],
              expected_outputs=["data/processed/corpus.csv"])
    _run_step(["python", "scripts/03_split_dataset.py"],
              expected_outputs=["data/processed/train.csv",
                                "data/processed/val.csv",
                                "data/processed/test.csv"])
    print("✓ Data prepared.")
else:
    print("⏭  Skipping data prep (cached or templates_only).")


### 7b.2 Train RoBERTa-Tagalog + DistilBERT-multilingual (scripts 04 / 05 / 17)

In [ ]:
if os.environ.get("_NEED_TRAIN") == "1":
    # Script 17 wraps scripts 04 and 05 with the multi-seed protocol from the paper
    _run_step(
        ["python", "scripts/17_run_5seeds_detectors.py",
         "--run_id", TRAIN_RUN_ID,
         "--seeds", TRAIN_SEEDS],
        expected_outputs=[]  # produces models/<task>/run_<id>/seed_<n>/ — variable structure
    )
    # Pick the best seed and copy to canonical location (scripts/04 and 05 expected paths)
    try:
        _run_step(
            ["python", "scripts/export_best_detectors_by_val.py",
             "--run_id", TRAIN_RUN_ID],
            expected_outputs=["models/roberta_finetuned/config.json",
                              "models/distilbert_multilingual_finetuned/config.json"]
        )
    except SystemExit:
        print("  (export_best_detectors_by_val.py not found or failed — falling back)")
        # Fallback: train each detector once at canonical location
        _run_step(["python", "scripts/04_train_robertaMINERVA.py"],
                  expected_outputs=["models/roberta_finetuned/config.json"])
        _run_step(["python", "scripts/05_train_distilbertMINERVA.py"],
                  expected_outputs=["models/distilbert_multilingual_finetuned/config.json"])
    print("✓ Detectors trained.")
else:
    print("⏭  Skipping detector training (cached or templates_only).")


#### 7b.2.1 Seed-statistics summary

Reads `reports/detector_seed_stats.json` (written by script 17) and prints panel-ready stats: mean ± std + paired t-test for RoBERTa vs DistilBERT.

In [ ]:
# v2.9.3: panel-ready seed statistics
import json
from pathlib import Path

_p = Path('reports/detector_seed_stats.json')
if _p.exists():
    stats = json.load(open(_p))
    print('━' * 60)
    print(' Detector seed statistics (n =', stats['n_seeds'], 'seeds)')
    print('━' * 60)
    print(f"  RoBERTa-Tagalog test F1:        {stats['roberta_test_f1_mean']*100:.2f}% ± {stats['roberta_test_f1_std']*100:.2f}%")
    print(f"  DistilBERT-multilingual test F1: {stats['distilbert_test_f1_mean']*100:.2f}% ± {stats['distilbert_test_f1_std']*100:.2f}%")
    if 'paired_ttest' in stats:
        pt = stats['paired_ttest']
        if 'p_value' in pt:
            print(f"  Paired t-test (RoBERTa vs DistilBERT):")
            print(f"    t = {pt['t_statistic']:.3f},  p = {pt['p_value']:.4f}")
            print(f"    {pt['interpretation']}")
        else:
            print(f"  Paired t-test: {pt.get('skipped', 'N/A')}")
    print('━' * 60)
else:
    print('⏭ No seed-stats report — only relevant after detector training runs (PIPELINE_MODE != templates_only).')


### 7b.3 Extract features (script 06)

In [ ]:
if os.environ.get("_NEED_TRAIN") == "1":
    _run_step(["python", "scripts/06_extract_features.py"],
              expected_outputs=["data/features/train_tabular.csv",
                                "data/features/val_tabular.csv",
                                "data/features/test_tabular.csv"])
    print("✓ Features extracted.")
else:
    print("⏭  Skipping feature extraction (cached or templates_only).")


### 7b.4 Train QLattice symbolic regression (script 08)

In [ ]:
if os.environ.get("_NEED_TRAIN") == "1":
    _run_step(["python", "scripts/08_train_qlattice.py"],
              expected_outputs=["models/qlattice_equation.txt"])
    print("✓ QLattice trained.")
    print("\n  Equation:")
    print("  " + Path("models/qlattice_equation.txt").read_text(encoding="utf-8").strip())
else:
    print("⏭  Skipping QLattice training (cached or templates_only).")
    if Path("models/qlattice_equation.txt").exists():
        print("\n  Cached QLattice equation:")
        print("  " + Path("models/qlattice_equation.txt").read_text(encoding="utf-8").strip())


### 7b.5 Train Dual-Embedding GNN (script 09)

In [ ]:
if os.environ.get("_NEED_TRAIN") == "1":
    _run_step(["python", "scripts/09_train_degnn.py"],
              expected_outputs=["data/features/degnn_preds.csv"])
    print("✓ DE-GNN trained.")
else:
    print("⏭  Skipping DE-GNN training (cached or templates_only).")


### 7b.6 JCBlaise test-set evaluation (script 15)

In [ ]:
EVAL_REPORT = "logs/eval_detectors_report.json"
TEST_CSV = "data/processed/test.csv"

if os.environ.get("_NEED_TRAIN") == "1":
    # Verify upstream artifacts exist before invoking script 15
    if not Path(TEST_CSV).exists():
        print(f"⚠  Cannot run held-out evaluation: {TEST_CSV} is missing.")
        print(f"    This means scripts 01/02/03 (data prep) didn't complete successfully.")
        print(f"    Re-run cell 7b.1 above, or set PIPELINE_MODE='templates_only'.")
    else:
        _run_step(
            ["python", "scripts/15_evaluate_detectors.py",
             "--test_csv", TEST_CSV,
             "--out_json", EVAL_REPORT],
            expected_outputs=[EVAL_REPORT]
        )
        print("✓ Detector evaluation complete.")
        import json as _json
        rep = _json.load(open(EVAL_REPORT))
        print(_json.dumps(rep.get("summary", rep), indent=2))
        # Cache to Drive for next session
        _save_to_drive()
elif PIPELINE_MODE != "templates_only" and Path(EVAL_REPORT).exists():
    print("Cached evaluation report:")
    import json as _json
    rep = _json.load(open(EVAL_REPORT))
    print(_json.dumps(rep.get("summary", rep), indent=2))
else:
    print("⏭  Skipping evaluation (templates_only).")


### 7b checkpoint: neuro-symbolic signals ready

If you ran in `full_cached` or `full_retrain` mode, you should now have:

## 8. Generate template cards (script 30)

The 18 templates produce 900 raw cards (18 × 18 per-template × 3 archetypes when applicable).

In [ ]:
import os
os.environ["N_PER_TEMPLATE"] = str(N_PER_TEMPLATE)

!python scripts/30_template_scenario_generator.py \
    --out_file generated/template_cards.json \
    --n_per_template $N_PER_TEMPLATE \
    --report_out reports/template_gen.json


## 8b. (Optional) GPT-2 neuro-symbolic augmentation, gated on `USE_GPT2`

This runs the neuro-symbolic conditioning pipeline (Mode B, introduced in, refined through):

In [ ]:
import json

if USE_GPT2:
    print("=" * 60)
    print("v2.9.0: NEURO-SYMBOLIC GPT-2 PATH (BATB SO 2)")
    print("=" * 60)

    needed = [
        "data/processed/train.csv",
        "data/processed/val.csv",
        "data/features/train_tabular.csv",
        "data/features/val_tabular.csv",
        "data/features/degnn_preds.csv",
        "models/qlattice_equation.txt",
    ]
    missing = [p for p in needed if not Path(p).exists()]
    if missing:
        print(f"⚠ Missing upstream artifacts: {missing}")
        print(f"  Section 7b above produces these. Set PIPELINE_MODE='full_*'")
        print(f"  Skipping GPT-2 path — pipeline continues with templates only.")
    else:
        # ----------------------------------------------------------------
        # 10b: build neuro-symbolic corpus (percentile binning by default)
        # ----------------------------------------------------------------
        _run_step(
            ["python", "scripts/10b_prepare_gpt2_neurosymbolic.py",
             "--train_csv", "data/processed/train.csv",
             "--val_csv", "data/processed/val.csv",
             "--train_features", "data/features/train_tabular.csv",
             "--val_features", "data/features/val_tabular.csv",
             "--degnn_preds", "data/features/degnn_preds.csv",
             "--qlattice_equation", "models/qlattice_equation.txt",
             "--out_dir", "data/gpt2_neurosymbolic",
             "--report_out", "reports/gpt2_neurosymbolic_corpus.json"],
            expected_outputs=["data/gpt2_neurosymbolic/train.txt",
                              "data/gpt2_neurosymbolic/val.txt"]
        )

        # ----------------------------------------------------------------
        # 11b: fine-tune
        # ----------------------------------------------------------------
        _run_step(
            ["python", "scripts/11b_train_gpt2_neurosymbolic.py",
             "--corpus_dir", "data/gpt2_neurosymbolic",
             "--base_model", GPT2_BASE_MODEL,
             "--out_dir", "models/gpt2_tagalog_neurosymbolic",
             "--epochs", str(GPT2_EPOCHS),
             "--lr", str(GPT2_LR)],
            expected_outputs=["models/gpt2_tagalog_neurosymbolic/config.json"]
        )

        # ----------------------------------------------------------------
        # 12b + 13: PERSISTENT GENERATION LOOP (v2.9.0)
        # Generate a batch, score it, count how many would survive merge,
        # retry with a new seed if not enough — up to GPT2_MAX_ATTEMPTS.
        # ----------------------------------------------------------------
        def _count_promoted_for_label(accumulator_path, label):
            """Run script 29 in dry mode against `accumulator_path` only
            (empty templates, empty other-label) and return promoted count."""
            empty_other = Path("/tmp/_empty.jsonl"); empty_other.touch()
            empty_templates = Path("/tmp/_empty_templates.json")
            empty_templates.write_text("[]")
            dry_out = Path("/tmp/_dry_merged.json")
            dry_report = Path("/tmp/_dry_report.json")
            cmd = ["python", "scripts/29_merge_gpt2_into_pool.py",
                   "--templates", str(empty_templates),
                   "--gpt2_fake", str(accumulator_path) if label == "fake" else str(empty_other),
                   "--gpt2_real", str(accumulator_path) if label == "real" else str(empty_other),
                   "--out", str(dry_out),
                   "--report_out", str(dry_report)]
            _run_step(cmd, expected_outputs=[str(dry_report)])
            rep = json.loads(dry_report.read_text())
            return int(rep.get("gpt2_promoted_by_label", {}).get(label, 0))

        def _generate_until_target(label, base_seed):
            accumulator = Path(f"generated/gpt2_neuro_final_{label}.jsonl")
            if accumulator.exists():
                accumulator.unlink()
            accumulator.touch()

            n_promoted = 0
            for attempt in range(GPT2_MAX_ATTEMPTS):
                seed = base_seed + attempt
                raw = f"generated/_a{attempt}_{label}_raw.jsonl"
                scored = f"generated/_a{attempt}_{label}_scored.jsonl"
                final = f"generated/_a{attempt}_{label}_final.jsonl"

                _run_step(
                    ["python", "scripts/12b_generate_gpt2_neurosymbolic.py",
                     "--target", label, "--n", str(GPT2_BATCH_SIZE),
                     "--seed", str(seed),
                     "--max_new_tokens", str(GPT2_MAX_NEW_TOKENS),
                     "--gen_model_dir", "models/gpt2_tagalog_neurosymbolic",
                     "--graph", "high", "--qlat", "high", "--ensem", "high",
                     "--tier", "novice",
                     "--out", raw],
                    expected_outputs=[raw]
                )
                _run_step(
                    ["python", "scripts/13_score_generated_with_qlattice.py",
                     "--in_jsonl", raw, "--target", label,
                     "--out_scored", scored, "--out_final", final],
                    expected_outputs=[final]
                )

                with open(final, "r", encoding="utf-8") as fi, \
                     open(accumulator, "a", encoding="utf-8") as fo:
                    fo.writelines(fi)

                n_promoted = _count_promoted_for_label(accumulator, label)
                print(f"  [{label}] attempt {attempt+1}/{GPT2_MAX_ATTEMPTS} "
                      f"(seed {seed}): {n_promoted} promoted so far "
                      f"(target {GPT2_MIN_PROMOTED_PER_LABEL})")

                if n_promoted >= GPT2_MIN_PROMOTED_PER_LABEL:
                    print(f"  ✓ [{label}] target reached, stopping retries.")
                    break
            else:
                print(f"  ⚠ [{label}] max attempts reached. "
                      f"Got {n_promoted} promoted (target was {GPT2_MIN_PROMOTED_PER_LABEL}).")
            return n_promoted

        print()
        print("─" * 60)
        print("Persistent generation: target ≥{} per label, max {} attempts."
              .format(GPT2_MIN_PROMOTED_PER_LABEL, GPT2_MAX_ATTEMPTS))
        print("─" * 60)

        n_fake = _generate_until_target("fake", base_seed=13)
        n_real = _generate_until_target("real", base_seed=27)

        print()
        print(f"✓ Neuro-symbolic GPT-2 generation complete.")
        print(f"  fake: {n_fake} cards in generated/gpt2_neuro_final_fake.jsonl")
        print(f"  real: {n_real} cards in generated/gpt2_neuro_final_real.jsonl")
        print(f"  These will be merged with template cards by script 29 (next cell).")
else:
    print("⏭  Skipping GPT-2 path (USE_GPT2=False).")


## 8b.5 Merge GPT-2 cards into the template stream (script 29)

In [ ]:
if USE_GPT2 and Path('generated/gpt2_neuro_final_fake.jsonl').exists():
    _run_step(
        ['python', 'scripts/29_merge_gpt2_into_pool.py',
         '--templates', 'generated/template_cards.json',
         '--gpt2_fake', 'generated/gpt2_neuro_final_fake.jsonl',
         '--gpt2_real', 'generated/gpt2_neuro_final_real.jsonl',
         '--out', 'generated/template_plus_gpt2_cards.json',
         '--report_out', 'reports/merge_gpt2_into_pool.json',
         '--require_at_least', str(GPT2_MIN_PROMOTED_PER_LABEL * 2)],
        expected_outputs=['generated/template_plus_gpt2_cards.json',
                          'reports/merge_gpt2_into_pool.json']
    )
    _MERGED_INPUT_PATH = 'generated/template_plus_gpt2_cards.json'
else:
    print('⏭  GPT-2 outputs not present — pipeline continues with templates only.')
    _MERGED_INPUT_PATH = 'generated/template_cards.json'

print(f'\nNext stage will use: {_MERGED_INPUT_PATH}')


## 9. Pseudonymize people (script 31)

Catches any real-name leaks in the cards. For template-generated cards, this is

In [ ]:
# v2.9.0: read merged file (templates + GPT-2 promotions) if it exists,
# else fall back to template-only.
_in = _MERGED_INPUT_PATH if '_MERGED_INPUT_PATH' in dir() else 'generated/template_cards.json'
!python scripts/31_universal_pseudonymize.py \
    --in_file {_in} \
    --out_file generated/cards_pseudo.json \
    --report_out reports/pseudo.json


## 9b. Pseudonymize places (script 35)

In [ ]:
!python scripts/35_pseudonymize_places.py \
    --in_file generated/cards_pseudo.json \
    --out_file generated/cards_pseudo_places.json \
    --blocklist templates/places_blocklist.txt \
    --report_out reports/pseudo_places.json


## 10. Balance verdicts and candidates (script 21)

In [ ]:
os.environ["TARGET_POOL"] = str(TARGET_POOL)

!python scripts/21_balance_unity_cards.py \
    --in_file generated/cards_pseudo_places.json \
    --out_file generated/balanced.json \
    --target_total $TARGET_POOL \
    --report_out reports/balance.json


## 11. Election-theme filter (script 23)

Templates are on-theme by construction, so accept rate should be ~95%+.

In [ ]:
!python scripts/23_enforce_election_theme.py \
    --in_file generated/balanced.json \
    --out_file generated/themed.json \
    --report_out reports/theme.json \
    --rejection_log reports/theme_rej.jsonl


## 12. Curate teaching pool (script 24)

Enforces the credible-card-per-day quota and stratifies by tactic/tier.

In [ ]:
os.environ["DAYS_IN_DECK"] = str(DAYS_IN_DECK)
os.environ["CARDS_PER_DAY"] = str(CARDS_PER_DAY)
os.environ["MIN_CREDIBLE_PER_DAY"] = str(MIN_CREDIBLE_PER_DAY)
os.environ["RNG_SEED"] = str(RNG_SEED)

!python scripts/24_curate_teaching_cards.py \
    --in_file generated/themed.json \
    --out_file generated/unity_cards_pool.json \
    --reject_out generated/pool_rej.json \
    --report_out reports/pool.json \
    --target_pool_size $TARGET_POOL \
    --days $DAYS_IN_DECK --cards_per_day $CARDS_PER_DAY \
    --min_credible_per_day $MIN_CREDIBLE_PER_DAY \
    --seed $RNG_SEED


## 13. Draw per-user decks (script 28)

Eight demo players: alice, bob, charlie, diana, erika, fiona, greg, hana.

In [ ]:
!python scripts/28_draw_user_deck.py \
    --pool_file generated/unity_cards_pool.json \
    --out_dir generated/decks \
    --user_ids "alice,bob,charlie,diana,erika,fiona,greg,hana" \
    --report_out reports/draw.json


## 14. Faithfulness audit (script 26)

Every card's explanation phrases are re-checked against its `fired_indicators`.

In [ ]:
!python scripts/26_faithfulness_audit.py \
    --in_file generated/unity_cards_pool.json \
    --report_out reports/faith.json


## 15. Detector validation on templates (script 32)

Validates the synthetic detector scores against gold verdicts. With

In [ ]:
!python scripts/32_validate_detectors_on_templates.py \
    --pool_file generated/unity_cards_pool.json \
    --report_out reports/det.json \
    --markdown_out reports/det.md


## 16. Strict allowlist enforcer (script 33)

This is the closure on the user-reported issue: variety of real political names

In [ ]:
import os

# Use the candidate config (single source of truth)
CANDIDATE_PROFILES = "scripts/candidate_config.py"
BLOCKLIST = "templates/jcblaise_real_names_blocklist.txt"

print(f"Using candidate profiles: {CANDIDATE_PROFILES}")
print(f"Using blocklist file:     {BLOCKLIST}")
print()

# Run in REJECT mode - strictest, recommended for game export.
!python scripts/33_strict_name_allowlist.py \
    --in_file generated/unity_cards_pool.json \
    --candidate_profiles {CANDIDATE_PROFILES} \
    --blocklist_file {BLOCKLIST} \
    --out_file generated/unity_cards_pool_strict.json \
    --report_out reports/strict_allowlist.json \
    --mode reject

# Print key numbers from the report
import json as _json
_r = _json.load(open("reports/strict_allowlist.json"))
print()
print(f"Strict allowlist pass rate : {_r['pass_rate_pct']}%")
print(f"  Cards in    : {_r['n_input_cards']}")
print(f"  Cards out   : {_r['n_kept']}")
print(f"  Cards rejected : {_r['n_rejected']}")
if _r.get('top_foreign_names'):
    print(f"  Top foreign names found:")
    for x in _r['top_foreign_names'][:5]:
        print(f"    - {x['name']!r} ({x['reason']}) - {x['count']}x")
else:
    print(f"  No foreign names found - pool is clean!")


## 16b. (Optional) Held-out detector evaluation — deferred to external validators

> This block is now . Off-distribution

In [ ]:
# v2.9.6: holdout eval is OPTIONAL — only runs if CSV is hand-labeled.
# Canonical pipeline defers off-distribution validation to external Filipino
# news-verification services (see docs/HOLDOUT_VALIDATION_STRATEGY.md).
from pathlib import Path
import csv

_holdout = Path('templates/holdout_gpt2_labeled.csv')
_has_labels = False

if _holdout.exists():
    # Check if any rows have non-empty true_label
    try:
        with open(_holdout, newline='', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                lbl = (row.get('true_label') or '').strip().lower()
                if lbl in ('real', 'fake'):
                    _has_labels = True
                    break
    except Exception as e:
        print(f'⚠  Could not read {_holdout}: {e}')

if _has_labels:
    print('✓ Holdout CSV has labels — running internal off-distribution eval')
    !python scripts/37_holdout_detector_eval.py \
        --holdout templates/holdout_gpt2_labeled.csv \
        --report_out reports/holdout_detector_eval.json
else:
    print('⏭  Skipping internal holdout evaluation (v2.9.6 default).')
    print('   Reason: off-distribution validation is deferred to external Filipino')
    print('   news-verification services (Rappler/VERA/Tsek/AFP) and to the Thesis 3')
    print('   SHS pilot. The CSV scaffold remains in templates/ for optional use.')
    print('   See docs/HOLDOUT_VALIDATION_STRATEGY.md for the full rationale.')
    print()
    print('   Validation evidence for Thesis 2 defense comes from:')
    print('     - reports/detectors_5seed_summary_v28_panel.json  (5-seed F1 ± std)')
    print('     - reports/det.json                                 (JCBlaise test, with caveat)')
    print('     - reports/faith.json                               (faithfulness)')
    print('     - reports/strict_allowlist.json                    (name leakage)')
    print('     - reports/pool.json                                (diversity)')
    print('     - reports/draw.json                                (deck overlap)')

## 16c. Export pilot pack (script 40)

Generates a 30-card pack stratified across verdict, tier, candidate, and tactic.

In [ ]:
os.environ["PILOT_N"] = str(PILOT_N)
os.environ["PILOT_SEED"] = str(PILOT_SEED)

!python scripts/40_export_pilot_pack.py \
    --pool_file generated/unity_cards_pool_strict.json \
    --out_dir reports/pilot_pack \
    --n $PILOT_N --seed $PILOT_SEED


## 17. Final dashboard — verified metrics

Reads every report file and prints the consolidated check.

In [ ]:
import json, glob

def safe_load(p):
    try:
        return json.load(open(p, encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError):
        return None

print("=" * 70)
print("M.I.N.E.R.V.A. v2.6-final-consolidated + P1.2 — END-TO-END METRICS")
print("=" * 70)

pool = safe_load("generated/unity_cards_pool.json")
if pool:
    meta = pool.get("_metadata", {})
    print(f"\nPool size            : {meta.get('pool_size')} cards")
    print(f"  REAL/FAKE/UNCERTAIN : {meta.get('real_count')}/{meta.get('fake_count')}/{meta.get('uncertain_count')}")
    print(f"  Candidates          : {meta.get('candidate_distribution')}")
    print(f"  Tier distribution   : {meta.get('tier_distribution')}")
    ind = meta.get("indicator_coverage", {})
    print(f"  Indicator coverage  : {len([k for k,v in ind.items() if v > 0])}/12  -- {sorted(ind.keys())}")

faith = safe_load("reports/faith.json")
if faith:
    print(f"\nFaithfulness audit    : {faith.get('pass_rate')}%  ({faith.get('passed')}/{faith.get('total_audited')})")

draw = safe_load("reports/draw.json")
if draw:
    po = draw.get("pairwise_overlap", {})
    if po:
        print(f"Pairwise overlap      : mean {po.get('mean_overlap_pct')}%  range [{po.get('min_overlap_pct')}, {po.get('max_overlap_pct')}]")

det = safe_load("reports/det.json")
if det:
    pdm = det.get("per_detector_metrics", {}).get("p_ensemble_fake", {})
    if pdm:
        print(f"Detector ensemble     : acc={pdm.get('accuracy')}%  F1(FAKE)={pdm.get('f1_fake')}%  on n={pdm.get('n')}")

# Pilot pack
import os
pilot_dir = "reports/pilot_pack"
if os.path.isdir(pilot_dir):
    files = os.listdir(pilot_dir)
    print(f"\nPilot pack outputs ({pilot_dir}/):")
    for f in sorted(files):
        sz = os.path.getsize(os.path.join(pilot_dir, f))
        print(f"  {f}  ({sz:,} bytes)")

print("\n" + "=" * 70)
print("Targets:")
print("  Pool size 668 (±15)        Indicator coverage  12/12")
print("  Faithfulness audit  100%    Pairwise overlap   <13% mean")
print("  Detector ensemble   100%    Pilot pack          3 files")
print("=" * 70)


## 17b. Verify audit-remediation fixes landed

After the run completes, this cell verifies that the AND audit fixes are reflected in actual output:

In [ ]:
# v2.9.4 + v2.9.5: verify audit-remediation fixes landed in output
import json
from pathlib import Path

print("=" * 60)
print(" v2.9.4 + v2.9.5 audit-remediation verification")
print("=" * 60)

# Check 1: Explanation diversity (the P1 fix)
p = Path('reports/pool.json')
if p.exists():
    r = json.load(open(p))
    d = r.get('explanation_diversity', {})
    total = d.get('total', 0)
    unique = d.get('unique', 0)
    pct = 100 * unique / total if total > 0 else 0
    status = '✓ PASS' if pct >= 30 else '⚠ BELOW TARGET'
    print(f"\n[1] Explanation diversity:")
    print(f"    {unique}/{total} = {pct:.1f}%   {status}   (target ≥ 30%, v2.9.0 was 5.4%)")
else:
    print("\n[1] reports/pool.json not yet generated")

# Check 2: Version stamps in GPT-2 reports
print(f"\n[2] Version stamp harmonization (target: v2.9.4 or v2.9.0+):")
version_files = [
    'reports/gpt2_neurosymbolic_corpus.json',
    'reports/gpt2_neurosymbolic_training.json',
    'reports/gpt2_neurosymbolic_generation.json',
    'reports/pseudo_places.json',
    'reports/holdout_detector_eval.json',
]
for vf in version_files:
    p = Path(vf)
    if p.exists():
        v = json.load(open(p)).get('version', 'N/A')
        ok = v in ('v2.9.0', 'v2.9.1', 'v2.9.2', 'v2.9.3', 'v2.9.4', 'v2.9.5')
        status = '✓' if ok else '⚠ outdated'
        print(f"    {status} {vf}: version={v}")
    else:
        print(f"    -  {vf}: not generated")

# Check 3: Holdout detector eval
p = Path('reports/holdout_detector_eval.json')
if p.exists():
    r = json.load(open(p))
    n_total = r.get('n_total', 0)
    n_real = r.get('n_real', 0)
    n_fake = r.get('n_fake', 0)
    n_unc = r.get('n_uncertain_excluded', 0)
    print(f"\n[3] Held-out detector eval:")
    print(f"    n_total={n_total}, n_real={n_real}, n_fake={n_fake}, n_uncertain_excluded={n_unc}")
    if n_real > 0 or n_fake > 0:
        metrics = r.get('detector_metrics', {})
        for det in ('p_roberta_fake', 'p_distil_fake', 'p_ensemble_fake'):
            m = metrics.get(det, {})
            if m.get('n', 0) > 0:
                print(f"    {det}: F1={m.get('f1', 0)*100:.2f}%, acc={m.get('accuracy', 0)*100:.2f}%, n={m['n']}")
    else:
        print("    ⏭  Holdout not yet hand-labeled.")
        print("       Open templates/holdout_gpt2_labeled.csv and fill in the")
        print("       'true_label' column (real/fake/uncertain) for each row,")
        print("       then re-run scripts/37_holdout_detector_eval.py.")

# Check 4: Faithfulness + strict allowlist (preserved from v2.9.0)
print(f"\n[4] v2.9.0 audit-response numbers (should still be ceiling):")
for fname, label, target in [
    ('reports/faith.json', 'Faithfulness pass rate', 98),
    ('reports/strict_allowlist.json', 'Strict allowlist pass rate', 99),
]:
    p = Path(fname)
    if p.exists():
        r = json.load(open(p))
        rate = r.get('pass_rate', r.get('pass_rate_pct', 0))
        status = '✓' if rate >= target else '⚠'
        print(f"    {status} {label}: {rate:.1f}%  (target ≥ {target}%)")


# v2.9.5 Check A: GPT-2 training seed is not 42 (Picard 2021)
print(f"\n[5] (v2.9.5) GPT-2 training seed ≠ 42:")
p = Path('reports/gpt2_neurosymbolic_training.json')
if p.exists():
    r = json.load(open(p))
    seed_used = r.get('training', {}).get('seed') or r.get('hyperparameters', {}).get('seed')
    if seed_used is None:
        seed_used = r.get('seed', 'N/A')
    if isinstance(seed_used, int) and seed_used != 42:
        print(f"    ✓ seed={seed_used}  (target: not 42 per Picard 2021)")
    elif seed_used == 42:
        print(f"    ⚠ seed=42 (Picard 2021 cherry-picking risk; v2.9.5 default is 1729)")
    else:
        print(f"    -  seed={seed_used} (not int)")
else:
    print("    -  reports/gpt2_neurosymbolic_training.json not generated")

# v2.9.5 Check B: schema-invalid categorization present
print(f"\n[6] (v2.9.5) Schema-invalid categorization in balance.json:")
p = Path('reports/balance.json')
if p.exists():
    r = json.load(open(p))
    if 'schema_invalid_by_reason' in r:
        breakdown = r['schema_invalid_by_reason']
        print(f"    ✓ schema_invalid_by_reason field present:")
        for reason, count in sorted(breakdown.items(), key=lambda kv: -kv[1])[:5]:
            print(f"        {reason}: {count}")
    else:
        print("    ⚠ schema_invalid_by_reason missing (v2.9.5 fix not deployed)")
else:
    print("    -  reports/balance.json not generated")

# v2.9.5 Check C: det.json has the tautology caveat
print(f"\n[7] (v2.9.5) det.json tautology caveat:")
p = Path('reports/det.json')
if p.exists():
    r = json.load(open(p))
    metric_kind = r.get('metric_kind')
    has_interp = 'interpretation' in r
    if metric_kind == 'internal_consensus' and has_interp:
        print(f"    ✓ metric_kind='internal_consensus' + interpretation field present")
    else:
        print(f"    ⚠ metric_kind={metric_kind!r}, interpretation={'present' if has_interp else 'missing'}")
        print("       (v2.9.5 fix not deployed; det.json's 100% is tautological)")
else:
    print("    -  reports/det.json not generated")

print("\n" + "=" * 60)
print(" If all checks pass, v2.9.4 + v2.9.5 are fully realized in this run.")
print("=" * 60)


## 18. Sample 6 cards from a player's deck

Quick sanity check — read 6 cards as a SHS student would. Do they make sense?

In [ ]:
import json, random

decks_dir = "generated/decks"
deck_files = sorted([f for f in os.listdir(decks_dir) if f.endswith(".json")])
if not deck_files:
    print("⚠ No deck files in generated/decks/")
else:
    deck = json.load(open(os.path.join(decks_dir, deck_files[0]), encoding="utf-8"))
    cards = deck.get("cards", deck if isinstance(deck, list) else [])
    print(f"Player: {deck_files[0].replace('.json','')}")
    print(f"Deck size: {len(cards)}")
    print()
    random.seed(42)
    sample = random.sample(cards, min(6, len(cards)))
    for i, c in enumerate(sample, 1):
        print(f"--- Card {i} | {c.get('verdict')} | {c.get('candidate')} | {c.get('provenance',{}).get('tactic')} | {c.get('provenance',{}).get('tier')} ---")
        print(f"  {c.get('text','')[:300]}")
        print()


## 19. Sample of the pilot pack (P1.2)

Print the first 2 cards from the questionnaire so you can see what students will see.

In [ ]:
quest_path = "reports/pilot_pack/questionnaire.md"
if os.path.exists(quest_path):
    text = open(quest_path, encoding="utf-8").read()
    # Print first 2 cards (header + Card 1 + Card 2)
    cards_split = text.split("---")
    print("\n---".join(cards_split[:5]))
else:
    print("⚠ Pilot pack not generated. Run cell 16.")


## 20. Save outputs (to Drive if mounted, otherwise download as zip)

In [ ]:
import shutil
from datetime import datetime, timezone
from pathlib import Path

# v2.9.0: list of artifacts to bundle. We include the QLattice equation
# and structured metadata so reviewers can audit the symbolic-regression
# rule used for explainability and GPT-2 conditioning.
_QL_FILES = [
    "models/qlattice_equation.txt",
    "models/best_qlattice.json",
    "models/qlattice/best_qlattice.json",
]

ts = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

if DRIVE_MOUNT_OK:
    # Path A: Drive is mounted — copy directories to Drive
    target = Path(f"/content/drive/MyDrive/{DRIVE_OUTPUT_DIR}/{ts}")
    target.mkdir(parents=True, exist_ok=True)

    for src in ["generated", "reports"]:
        if not Path(src).exists():
            continue
        dst = target / src
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"  ✓ {src} → {dst}")

    # v2.9.0: bundle the QLattice equation files alongside the rest
    ql_target = target / "models"
    ql_target.mkdir(parents=True, exist_ok=True)
    for ql_file in _QL_FILES:
        if Path(ql_file).exists():
            dst = ql_target / Path(ql_file).name
            shutil.copy2(ql_file, dst)
            print(f"  ✓ {ql_file} → {dst}")
        else:
            print(f"  ⏭  {ql_file} not found")

    print(f"\n✓ All outputs saved to Drive: /content/drive/MyDrive/{DRIVE_OUTPUT_DIR}/{ts}/")

else:
    # Path B: Drive not mounted — bundle into a single zip and trigger browser download
    print("⚠ Drive not mounted. Bundling outputs into a zip for download...")

    bundle_name = f"minerva_outputs_{ts}"
    bundle_root = Path(f"/tmp/{bundle_name}")
    bundle_root.mkdir(parents=True, exist_ok=True)

    for src in ["generated", "reports"]:
        if not Path(src).exists():
            print(f"  ⏭  {src}/ not found — skipping")
            continue
        shutil.copytree(src, bundle_root / src)
        print(f"  ✓ Staged {src}/")

    # v2.9.0: stage the QLattice equation files into models/ in the zip
    ql_target = bundle_root / "models"
    ql_target.mkdir(parents=True, exist_ok=True)
    for ql_file in _QL_FILES:
        if Path(ql_file).exists():
            shutil.copy2(ql_file, ql_target / Path(ql_file).name)
            print(f"  ✓ Staged {ql_file}")
        else:
            print(f"  ⏭  {ql_file} not found — skipping")

    # Zip the staged bundle
    zip_path_no_ext = f"/tmp/{bundle_name}"
    archive_path = shutil.make_archive(zip_path_no_ext, "zip", root_dir=str(bundle_root))
    archive_size_mb = Path(archive_path).stat().st_size / (1024 * 1024)
    print(f"  ✓ Created {archive_path} ({archive_size_mb:.1f} MB)")

    # Trigger browser download (Colab only)
    try:
        from google.colab import files as _colab_files
        print(f"\n⬇  Triggering browser download of {Path(archive_path).name} ...")
        _colab_files.download(archive_path)
        print(f"\n✓ Download initiated. Check your browser's download tray.")
    except ImportError:
        # Not running in Colab — print the local path
        print(f"\n⚠  Not running in Google Colab — automatic download unavailable.")
        print(f"   The bundle is at: {archive_path}")
        print(f"   Outputs are also still at:")
        print(f"     /content/MINERVA/generated/")
        print(f"     /content/MINERVA/reports/")
        print(f"   Open the Files tab in Colab's left sidebar to download manually.")
    except Exception as e:
        # Network or other Colab download failure
        print(f"\n⚠  Download trigger failed: {e}")
        print(f"   Bundle is still available at: {archive_path}")
        print(f"   You can right-click it in the Files tab and select 'Download'.")


## (Optional, run-once) Refresh JCBlaise blocklist from source dataset

This cell downloads the actual JCBlaise dataset from HuggingFace and extracts

In [ ]:
# Optional — run once, then commit the output to your repo
import os

# Make sure datasets is installed (Colab usually has it)
!python -m pip install -q datasets 2>&1 | tail -1

# Run the extractor
!python scripts/34_extract_jcblaise_names.py \
    --out_file templates/jcblaise_real_names_blocklist.txt \
    --report_out reports/jcblaise_extraction.json \
    --min_count 3

# Show the report
print()
import json as _json
if os.path.exists("reports/jcblaise_extraction.json"):
    _r = _json.load(open("reports/jcblaise_extraction.json"))
    print(f"Rows processed     : {_r['rows_processed']:,}")
    print(f"Unique names       : {_r['unique_names_total']:,}")
    print(f"Names blocklisted  : {_r['names_blocklisted']:,}")
    print(f"Top 5 names in dataset:")
    for x in _r['top_50_names_overall'][:5]:
        print(f"  - {x['name']} ({x['count']:,} occurrences)")
